# AICG3D T4 使用教程（配套 kaggle_t4.ipynb）

这份 Notebook 和 `kaggle_t4.ipynb` 是配套的，分工如下：

- `kaggle_t4.ipynb`：只负责部署——装 torch、拉固定版 ComfyUI、拷本仓库 AICG3D、
  下载模型、启动 ComfyUI、给隧道地址。
- 本份 `tutorial.ipynb`：ComfyUI 开起来之后，怎么用、怎么核对、怎么保活、怎么下载。
  它不重新部署，不改配置，不提交生成。

先跑完 `kaggle_t4.ipynb` 第 1-5 格，拿到隧道地址，再按下面顺序看。
版本锁死：ComfyUI commit ab08f7b2caa6d4007ecde3762116c6193613f43b、
torch 2.11.0+cu130、SageAttention2 2.1.2 sm75 轮子（验证通过才开）。


## 第 1 步：核查 ComfyUI 是不是真的在跑（只读，不提交生成）

跑下面这格，如果返回队列 running/pending 数字和输出目录，说明后端在。
如果返回“队列接口不可用”，先回 `kaggle_t4.ipynb` 第 5 格看日志，不要直接在 ComfyUI 里乱点。


In [ ]:
# 核查格：只读 ComfyUI 队列和输出目录
import urllib.request, json, time
from pathlib import Path
COMFY = Path("/kaggle/temp/h3_project") / "ComfyUI"
OUT_DIR = COMFY / "output" / "video"
print("核查时间：", time.strftime("%Y-%m-%d %H:%M:%S"))
print("ComfyUI 目录存在：", COMFY.exists(), "，main.py 存在：", (COMFY / "main.py").exists())
try:
    with urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=5) as r:
        q = json.loads(r.read().decode())
    print("队列 running:", len(q.get("queue_running", [])), "pending:", len(q.get("queue_pending", [])))
except Exception as e:
    print("队列接口不可用（可能未启动或已离线）：", repr(e))
if OUT_DIR.exists():
    videos = sorted(OUT_DIR.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    print("输出视频数：", len(videos))
    for v in videos[-5:]:
        print(" -", v.name, v.stat().st_size, "bytes")
else:
    print("输出目录尚不存在（还没有生成出视频）。")


## 第 2 步：在 ComfyUI 里导入 13 号工作流

1. 打开隧道地址，看到左侧菜单、画布、Queue Prompt，才算真正开了。
   如果只看到 “Your request was blocked.”，那是 Cloudflare 边缘拦了你当前浏览器网络，
   换你自己的设备/网络再打开，后端不一定有问题。
2. Workflow → Open，选本仓库 `workflows/` 里的 **`13.AICG3D_参考无限生成视频.json`**。
   不要一上来选 `13.AICG3D_MiniMax H3 无限长视频生成工作流1.2.json`，
   那份接了 DMAD Sampler / DMAD Sigmas，需要另外装 ComfyUI-DMAD，而且是 4 步加速路线，
   最终镜头不要靠降步数。
3. 导入后先不要点 Queue，先按下面几步核对。


## 第 3 步：清理资源库，只留你要用的图（最容易踩坑的一步）

- 找到 MiniMax H3 Aicg 资源库（`AICG3D_H3MediaLoader`）节点。
- 把模板自带的示例图全部删掉，只留本次要用的图。
- 以 Shot 01 为例，只留两张，顺序不能反：图片 1 是首帧 Shot_01.jpg，
  图片 2 是角色参考 01________Yu_Luosheng.jpg。
- 提示词里用 `<Picture 1>`、`<Picture 2>` 引用，或者在资源库里点一张图，
  它会自动插到光标所在的那一段提示词里。
- 如果还有缺失的模板图（红色“素材缺失”），先清理掉再提交，否则校验会直接拒绝。


## 第 4 步：先定段落数——单段还是多段，这是 17 秒教训的关键

13 号工作流默认是：加载器 → 全局设置 → 段落1 → 段落2 → 段落3 → 最终合成视频。

只想拍一段（比如只要 10 秒 Shot 01），必须二选一，否则会多出时长：
- 删掉段落 2、段落 3，只留段落 1，再把段落 1 的 segment 直接接到最终合成视频；或者
- 不删节点，但把段落 1 的 segment 直接接到最终合成，段落 2、3 不接进合成链。

11 月 2 号那次 17.2 秒就是这样来的：段落 1 是 10 秒没错，但段落 2、3（各 3 秒、
模板示例提示词）还连在合成链上，10+3+3 再加衔接就成了 17 秒。
“已填 1 段”只说明剧本面板里填了 1 段，不代表画布上只有 1 段，一定要看连线。

想拍长视频就保留链式：每一段单独写提示词、单独定秒数，种子建议 fixed，
上一段的 segment 接到下一段的 previous_segment，衔接帧数在全局设置里调。


## 第 5 步：全局设置和加载器核对

全局设置（`AICG3D_H3SequenceGlobal`）：
- 分辨率 / 宽高比：所有段落共用。Shot 01 用 480P、16:9（对应 864×480），帧率 24 锁定。
- 衔接帧数：H3 栅格只有 5 / 22 / 39… 这几档，填 1~5 按 5 帧衔接，填 6~20 按 22 帧衔接。
- 衔接方式：默认尾帧续写（latent），无损最快；显存收尾默认释放缓存。
- 采样：res_multistep / simple，步数保持 20 步，Lightning 关，最终镜头不要降步数。
- 注意力加速：默认自动（Sage 优先），T4 上只有验证标记通过才真正走 Sage，否则回落 SDPA。

加载器（`AICG3D_H3Loader`）：
- Ref2VA：minimax_h3_ref2va_pruned_int8_convrot.safetensors
- 文本编码器：qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors
- 视频 VAE：minimax_h3_video_vae_fp16.safetensors（不要用 int8 那个）
- 音频 VAE：minimax_h3_audio_vae_fp32.safetensors
- LoRA：最终镜头选不使用；Turbo LoRA 只当草稿档。


## 第 6 步：写提示词（Shot 01 修正版，可直接参考）

提示词写在分段提示词解析（`AICG3D_H3PromptSegmentSplit`）里，
点“一键填入全部视频段落”后会分发到各个段落节点，也可以直接在段落节点里改。

> The young man wearing the faded dark charcoal hoodie stands on the rain-soaked
> London city street outside the glowing esports arena at night, then slowly lifts
> his head, lowers his hands from his hood and turns slightly toward the arena.
> Heavy raindrops fall through the cool cyan and amber neon lights in slow motion.
> The camera smoothly dollys back and tilts up slightly, keeping the man prominent
> and in sharp focus, with clear facial features, as reflections ripple in the dark
> asphalt puddles. Distant pedestrians walk naturally on the wet ground, feet firmly
> planted, realistic gait, shadows and reflections aligned with their movement.
> 35mm anamorphic cinema film, realistic lighting, 24fps.
> Avoid blurry, low resolution, out of focus, distorted face, unclear person,
> floating people, sliding feet, unrealistic motion, static dull pose.

注意：H3 没有真正的负面提示词框，上面最后那句 Avoid 是当成正向提示词的一部分写进去的。
旁白（比如“曾站在世界之巅的人，也会被迫选择遗忘……”）是单独文案，不是生成提示词，
不要自动配音、不要塞进提示词里。


## 第 7 步：点 Queue 之前，先跑一遍核对清单（在心里打勾，不要跳过）

- 资源库里只有本次要用的图，顺序是首帧在前、角色参考在后，没有缺失图
- 段落数就是你要的段数：单段就只留段落 1 接合成，多段就按顺序链好
- 每段秒数、提示词、种子都核对过，种子是 fixed 不是 random（除非故意抽卡）
- 全局设置是 480P / 16:9 / 24fps，步数 20，Lightning 关
- 加载器模型、VAE、文本编码器都是第 5 步那几个文件名
- 注意力是自动（Sage 优先），T4 上验证标记在，失败了就接受 SDPA，不要硬开
- 队列是 Idle、没有重复提交的任务（提交前先看 Queue 面板）

核对完再点 Queue Prompt。生成期间每 20 分钟左右跑一次下面第 8 步的保活格，
Kaggle 超过 40 分钟无改动会弹“你还在吗”，点了保持运行才能继续。


In [ ]:
# 保活格：生成期间每 20 分钟跑一次，不提交生成、不改参数
import urllib.request, json, time
from pathlib import Path
COMFY = Path("/kaggle/temp/h3_project") / "ComfyUI"
print("保活核查时间：", time.strftime("%Y-%m-%d %H:%M:%S"))
try:
    with urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=5) as r:
        q = json.loads(r.read().decode())
    print("队列 running:", len(q.get("queue_running", [])), "pending:", len(q.get("queue_pending", [])))
except Exception as e:
    print("队列接口不可用：", repr(e))
print("本格只核查和保活。如果 Kaggle 弹出“你还在吗”，选保持会话运行，不要点关闭会话。")


## 第 8 步：生成完，先核对再下载（不要只看文件存在就当成功）

跑下面这格，它会列出输出视频、ffprobe 查分辨率/帧率/帧数/时长、
并把最新视频复制到 /kaggle/working/ 方便下载。
10 秒应该是 864×480、24fps、243 帧、约 10.125 秒。
核对完立刻下载到本地，不要只留在 Kaggle 上——上一轮 10 秒视频就是只留在 Kaggle 上，
会话离线后找不回来了。本地播放再核对一次：画面、人物脸、远处行人脚步、帧间变化，
全黑或静止就判失败，先查日志和工作流，不要把黑视频当成功。


In [ ]:
# 输出核对格：ffprobe + 复制到 /kaggle/working/（不要只看文件存在）
import subprocess, shutil
from pathlib import Path
COMFY = Path("/kaggle/temp/h3_project") / "ComfyUI"
OUT_DIR = COMFY / "output" / "video"
WORKING_OUT = Path("/kaggle/working")
if not OUT_DIR.exists():
    raise RuntimeError(f"输出目录不存在：{OUT_DIR}。请先在 ComfyUI 里生成并等队列完成。")
videos = sorted(OUT_DIR.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not videos:
    raise RuntimeError("输出目录里没有 mp4 文件。请先等生成完成。")
for v in videos:
    print(f"\n=== {v} ({v.stat().st_size} bytes) ===")
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                        "-show_entries", "stream=width,height,avg_frame_rate,codec_name,nb_frames,duration",
                        "-show_entries", "format=duration,size",
                        "-of", "default=noprint_wrappers=1", str(v)],
                       capture_output=True, text=True)
    print(r.stdout)
    print(r.stderr)
    dst = WORKING_OUT / v.name
    if dst != v:
        shutil.copyfile(v, dst)
        print(f"已复制到：{dst} ({dst.stat().st_size} bytes)，请尽快下载到本地。")
print("\n核对提示：画面是不是你要的场景和人物、帧与帧之间有没有变化、")
print("如果全黑或静止，说明可能是显存分片或 fp16_fix 没激活的问题，")
print("要先查日志和工作流，不要把黑视频当成成功结果。")


## 常见问题速查

- Sage 日志满屏 PassManager 失败但最后出片了：官方 Sage 在 sm75 上静默回落，等于没加速。
  本仓库修复后验证不过会自动走 SDPA，不应该再满屏刷这个错；如果还刷，先确认 custom_nodes 里
  是不是本仓库拷过去的那份 accel.py。
- 导入原生工作流提示“部分节点需要更新 ComfyUI”，列出 ComfyMathExpression：
  那是原生工作流的问题，不是 AICG3D 工作流的问题。AICG3D 13 号工作流不用这个节点。
- 提交后被校验拒绝、视频解码长边为 0：资源库里还有模板示例图缺失，先清理到只剩本次的图。
- 生成出来比你要的长（要 10 秒出了 17 秒）：段落 2、3 还连在合成链上，按第 4 步断开。
- OOM：highvram → normalvram → lowvram 降档，记录是哪一档跑通的。
- 隧道打开是 “Your request was blocked.”：Cloudflare 边缘拦了你当前浏览器网络，换设备/网络再试。
- Kaggle 会话突然离线：超过 40 分钟无改动，只能从头重建，所以长生成一定要每 20 分钟保活。
- 磁盘空间接近上限警告：模型在 /kaggle/temp 里约 67GB，目前不阻断运行，生成完尽快下载并清理旧输出。

完整文字版教程见仓库里的 `使用教程.md`，和本 Notebook 内容一致。
